# Cost Function and Gradient Descent


In [ ]:
import numpy as np
import math, copy
import matplotlib.pyplot as plt


## Problem Statement

You would like a model which can predict housing prices given the size of the house. Let’s use the following data:

| Size (1000 sqft) | Price (INR in Lakhs) |
|---:|---:|
| 1 | 30 |
| 2 | 50 |
| 3 | 68 |
| 4 | 75 |
| 5 | 100 |


In [ ]:
x_train = np.array([1.0, 2.0, 3.0, 4.0, 5.0])  # size in 1000s of square feet
y_train = np.array([30.0, 50.0, 68.0, 75.0, 100.0])  # price in INR Lakhs


## Computing Cost

The term **cost** is a measure of how well the model predicts the target house price; it is not the housing price itself.

For one feature, the cost is:

$$J(\theta_0, \theta_1) = \frac{1}{2m} \sum_{i=0}^{m-1}(h(x^{(i)}) - y^{(i)})^2$$

where

$$h(x^{(i)}) = \theta_0 + \theta_1x^{(i)}$$

- $h(x^{(i)})$ is the prediction for example $i$.
- $(h(x^{(i)}) - y^{(i)})^2$ is the squared prediction error.
- The errors are summed and divided by $2m$.
- $\theta_0$ is also called the bias ($b$); $\theta_1$ is the weight ($w$).


The code below calculates cost by looping over each example. In each loop:

- `h_x`, a prediction, is calculated.
- The difference between the target and prediction is calculated and squared.
- The result is added to the total cost.


In [ ]:
def compute_cost(x, y, b, w):
    """Computes the cost function for linear regression."""
    # Number of training examples
    m = x.shape[0]
    cost_sum = 0

    for i in range(m):
        # Calculate h(x), then the squared cost for this example.
        h_x = w * x[i] + b
        cost = (h_x - y[i]) ** 2
        cost_sum = cost_sum + cost

    total_cost = (1 / (2 * m)) * cost_sum
    return total_cost


## Gradient Descent Summary

The linear model predicts:

$$h(x^{(i)}) = wx^{(i)} + b$$

In linear regression, the parameters $w$ and $b$ are fitted by minimizing the cost over all training samples:

$$J(w,b) = \frac{1}{2m} \sum_{i=0}^{m-1}(h(x^{(i)}) - y^{(i)})^2$$


Gradient descent repeats the following updates until convergence:

$$\begin{align*}
w &:= w - \alpha \frac{\partial J(w,b)}{\partial w}\\
b &:= b - \alpha \frac{\partial J(w,b)}{\partial b}
\end{align*}$$

The parameters are updated **simultaneously**. First calculate all partial derivatives, then update both parameters. The derivatives are:

$$\frac{\partial J(w,b)}{\partial w} = \frac{1}{m} \sum_{i=0}^{m-1}(h(x^{(i)}) - y^{(i)})x^{(i)}$$

$$\frac{\partial J(w,b)}{\partial b} = \frac{1}{m} \sum_{i=0}^{m-1}(h(x^{(i)}) - y^{(i)})$$


## Implement Gradient Descent

Three functions are used:

- `compute_gradient` implements the two partial derivatives.
- `compute_cost` implements the cost function.
- `gradient_descent` uses both functions to fit $w$ and $b$.

The naming convention is `dj_db` for $\frac{\partial J}{\partial b}$ and `dj_dw` for $\frac{\partial J}{\partial w}$. “w.r.t.” means **with respect to**.


In [ ]:
def compute_gradient(x, y, w, b):
    """Computes the gradient for one-variable linear regression."""
    m = x.shape[0]
    dj_dw = 0
    dj_db = 0

    for i in range(m):
        h_x = w * x[i] + b
        dj_dw_i = (h_x - y[i]) * x[i]
        dj_db_i = h_x - y[i]
        dj_dw += dj_dw_i
        dj_db += dj_db_i

    dj_dw = dj_dw / m
    dj_db = dj_db / m
    return dj_dw, dj_db


### Gradient Descent

Now that gradients can be computed, use the update rule above to find optimal values of $w$ and $b$ on the training data.


In [ ]:
def gradient_descent(x, y, w_in, b_in, alpha, num_iters, cost_function, gradient_function):
    """Fits w and b by taking num_iters gradient-descent steps."""
    w = copy.deepcopy(w_in)  # avoid modifying a mutable input
    b = b_in
    J_history = []
    p_history = []

    for i in range(num_iters):
        # Calculate both gradients before updating either parameter.
        dj_dw, dj_db = gradient_function(x, y, w, b)

        # Update parameters using gradient descent.
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        # Save cost and parameter values for graphing later.
        if i < 100000:  # prevent resource exhaustion
            J_history.append(cost_function(x, y, b, w))
            p_history.append([w, b])

        # Print progress ten times, or once per iteration if fewer than ten.
        if i % math.ceil(num_iters / 10) == 0:
            print(f'Iteration {i:4}: Cost {J_history[-1]:0.2e} ',
                  f'dj_dw: {dj_dw: 0.3e}, dj_db: {dj_db: 0.3e}  ',
                  f'w: {w: 0.3e}, b:{b: 0.5e}')

    return w, b, J_history, p_history


In [ ]:
# Initialize parameters
w_init = 0
b_init = 0

# Gradient-descent settings
iterations = 10000
tmp_alpha = 1.0e-2

# Run gradient descent
w_final, b_final, J_hist, p_hist = gradient_descent(
    x_train, y_train, w_init, b_init, tmp_alpha, iterations,
    compute_cost, compute_gradient
)
print(f'(w,b) found by gradient descent: ({w_final:8.4f},{b_final:8.4f})')


### Cost versus iterations of gradient descent

A cost-versus-iteration plot measures progress. In a successful run, cost decreases. The initial decline is much faster than the final decline, so the plots use different iteration ranges.


In [ ]:
# Plot cost versus iteration
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12, 4))
ax1.plot(J_hist[:100])
ax2.plot(1000 + np.arange(len(J_hist[1000:])), J_hist[1000:])
ax1.set_title('Cost vs. iteration (start)')
ax2.set_title('Cost vs. iteration (end)')
ax1.set_ylabel('Cost'); ax2.set_ylabel('Cost')
ax1.set_xlabel('iteration step'); ax2.set_xlabel('iteration step')
plt.show()


### Predictions

Use the learned $w$ and $b$ in the hypothesis equation to predict housing values. The model is trained using thousands of square feet, so 1000 sqft is `1.0`, 2000 sqft is `2.0`, and 1750 sqft is `1.75`.


In [ ]:
# Use the hypothesis equation and learned parameters for predictions.
prediction_1000 = w_final * 1.0 + b_final
prediction_2000 = w_final * 2.0 + b_final
prediction_1750 = w_final * 1.75 + b_final

print(f'1000 sqft house prediction {prediction_1000:0.1f} INR Lakhs')
print(f'2000 sqft house prediction {prediction_2000:0.1f} INR Lakhs')
print(f'1750 sqft house prediction {prediction_1750:0.1f} INR Lakhs')

# Expected numerical values:
# 1000 sqft: 31.6
# 2000 sqft: 48.1
# 1750 sqft: 44.0
